## Build Style Vectors — Llama-3.2-3B-Instruct / 1B-Instruct + Prompt-Guard-86M

Place these files in the **same directory**:
```
main.py
build_vectors_llama32.ipynb
```

### Model settings reference
| Setting | Llama-3.1-8B | **Llama-3.2-3B** | **Llama-3.2-1B** | Prompt-Guard-86M |
|---|---|---|---|---|
| `LOCAL_MODEL_NAME` | Llama-3.1-8B-Instruct | **Llama-3.2-3B-Instruct** | **Llama-3.2-1B-Instruct** | Prompt-Guard-86M |
| `STEER_LAYER` | 20 | **16** | **10** | N/A (classifier) |
| `STEER_ALPHA` | 13.0 | **12.0** | **10.0** | N/A |
| Hidden dim | 4096 | **3072** | **2048** | 768 |
| Total layers | 32 | **28** | **16** | 12 |
| VRAM needed | ~16 GB | **~6 GB** | **~2 GB** | **<1 GB** |
| Prompt format | `<|begin_of_text|>` | **same** | **same** | `[CLS]` |
| EOS tokens | 2 | **2** | **2** | N/A |
| `add_special_tokens` | False | **False** | **False** | True |

### Runtime
- **3B model**: T4 (15 GB) or better
- **1B model**: free Colab T4 or even CPU (slow)
- **Prompt-Guard-86M**: runs on CPU in seconds

### Run order
**Cell 0a → Cell 0b → Cell 1 → … → Cell 11** (always start with 0a to clear VRAM)

For Prompt-Guard only: Cell 0a → Cell 1 → Cell 2 → Cell 3 → **Cell 12** (skip 6-11)


In [1]:
# ─── Cell 0a — AGGRESSIVE GPU CLEANUP (RUN THIS FIRST EVERY SESSION) ─────────
#
# Clears any previous model before loading a new one.
# If VRAM > 1 GB after this cell: Runtime → Restart runtime → re-run from Cell 0a.

import gc, sys

if 'main_module' in dir():
    try:
        main_module._model_cache.clear()
        print('  ✓ main_module._model_cache.clear() — done')
    except Exception as e:
        print(f'  cache clear skipped: {e}')

_gone = []
for _v in [k for k in list(globals().keys())
           if any(x in k.lower() for x in ['model', 'tokenizer', 'main_module'])]:
    try:
        del globals()[_v]
        _gone.append(_v)
    except Exception:
        pass
if _gone:
    print(f'  ✓ Deleted globals: {_gone}')

_unloaded = [k for k in list(sys.modules.keys()) if 'main_module' in k]
for _m in _unloaded:
    del sys.modules[_m]
if _unloaded:
    print(f'  ✓ Unloaded from sys.modules: {_unloaded}')

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        total     = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'\n  GPU         : {torch.cuda.get_device_name(0)}')
        print(f'  Total VRAM  : {total:.1f} GB')
        print(f'  Allocated   : {allocated:.2f} GB  <- should be ~0.0 GB')
        print(f'  Reserved    : {reserved:.2f} GB')
        print(f'  Approx free : {total - reserved:.1f} GB')
        print()
        if allocated > 1.0:
            print('  ⚠  WARNING: VRAM still in use — restart runtime.')
        else:
            print('  ✓ VRAM clean — safe to load model.')
    else:
        print('  ⚠  No GPU — Llama-3.2-1B and Prompt-Guard still work on CPU.')
except ImportError:
    print('  torch not installed yet — run Cell 1 first')



  GPU         : NVIDIA RTX 6000 Ada Generation
  Total VRAM  : 50.9 GB
  Allocated   : 0.00 GB  <- should be ~0.0 GB
  Reserved    : 0.00 GB
  Approx free : 50.9 GB

  ✓ VRAM clean — safe to load model.


In [2]:
# ─── Cell 0b — MOUNT GOOGLE DRIVE FOR PERSISTENT MODEL CACHE ─────────────────
#
# First run  : downloads weights to Drive
# Later runs : loads from Drive cache (much faster)

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_CACHE = '/content/drive/MyDrive/hf_model_cache'
    os.makedirs(DRIVE_CACHE, exist_ok=True)

    os.environ['HF_HOME']               = DRIVE_CACHE
    os.environ['TRANSFORMERS_CACHE']    = DRIVE_CACHE
    os.environ['HF_DATASETS_CACHE']     = DRIVE_CACHE
    os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_CACHE

    import glob
    existing = (glob.glob(f'{DRIVE_CACHE}/**/*.safetensors', recursive=True)
                + glob.glob(f'{DRIVE_CACHE}/**/*.bin', recursive=True))
    print(f'  ✓ HuggingFace cache -> {DRIVE_CACHE}')
    if existing:
        total_gb = sum(os.path.getsize(f) for f in existing) / 1e9
        print(f'  ✓ Found {len(existing)} cached weight file(s) ({total_gb:.1f} GB) — NO re-download needed!')
    else:
        print('  info: No cached weights yet — first run will download to Drive.')
except Exception as e:
    print(f'  Drive mount skipped ({e}) — using /root/.cache (re-downloads every session).')


  Drive mount skipped (No module named 'google.colab') — using /root/.cache (re-downloads every session).


In [3]:
# ─── Cell 1 — Install dependencies ──────────────────────────────────────────
# transformers>=4.45.0 required for Llama-3.2 and Prompt-Guard-86M
%pip install -q "transformers>=4.45.0" "pydantic>=2.9.0"

%pip install -q \
    fastapi==0.115.0 \
    uvicorn==0.30.6 \
    httpx==0.27.2 \
    python-dotenv>=1.0.0 \
    langchain-groq>=0.1.9 \
    langchain-core>=0.2.0 \
    rich>=13.7.0 \
    accelerate>=0.34.0 \
    huggingface_hub>=0.24.0 \
    scikit-learn>=1.4.0 \
    numpy>=1.26.0 \
    spacy

try:
    import torch
    print(f"✓ torch already available: {torch.__version__}")
except ImportError:
    !pip install -q "torch>=2.2.0"
    import torch

!python -m spacy download en_core_web_sm

import importlib, transformers
importlib.reload(transformers)
print(f"✓ transformers active version: {transformers.__version__}")
assert transformers.__version__ >= "4.45.0", \
    f"Still on {transformers.__version__} — RESTART KERNEL and rerun this cell."
print("✓ All dependencies installed")
# ── After installing, restart the kernel once, then re-run from Cell 1 ──────


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✓ torch already available: 2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 22.3 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
✓ transformers active version: 5.9.0
✓ All dependencies installed


In [ ]:
# ─── Cell 2 — Set secrets and authenticate with Hugging Face ─────────────────
#
# All three models (Llama-3.2-3B, 1B, Prompt-Guard-86M) are gated — HF_TOKEN required.

import os
from huggingface_hub import login

os.environ["HF_TOKEN"]     = "HF_TOKEN"   # https://huggingface.co/settings/tokens
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"    # https://console.groq.com

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN or HF_TOKEN == "hf_YOUR_TOKEN_HERE":
    raise EnvironmentError(
        "HF_TOKEN not set.\n"
        "Replace hf_YOUR_TOKEN_HERE with your actual token from huggingface.co/settings/tokens"
    )

login(token=HF_TOKEN)
print("✓ Hugging Face login successful")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if GROQ_API_KEY and GROQ_API_KEY != "gsk_YOUR_KEY_HERE":
    print("✓ GROQ_API_KEY found — full pipeline (Cell 13) will work")
else:
    print("⚠  GROQ_API_KEY not set — vector build/smoke-test still work; Cell 13 requires it")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Hugging Face login successful
✓ GROQ_API_KEY found — full pipeline (Cell 13) will work


In [5]:
# ─── Cell 3 — Select model and verify GPU ────────────────────────────────────
#
# Uncomment ONE model. Prompt-Guard-86M is a classifier; skip Cells 6-11, go to Cell 12.

import os, torch, transformers

# ── Pick your model (uncomment one) ──────────────────────────────────────────
SELECTED_MODEL = "meta-llama/Llama-3.2-3B-Instruct"    # ~6 GB VRAM, recommended
# SELECTED_MODEL = "meta-llama/Llama-3.2-1B-Instruct"  # ~2 GB VRAM, free T4
# SELECTED_MODEL = "meta-llama/Prompt-Guard-86M"        # classifier only, <1 GB

_DEFAULTS = {
    "meta-llama/Llama-3.2-3B-Instruct": {"layer": 16, "alpha": 12.0, "hidden": 3072, "vram_gb": 6.0},
    "meta-llama/Llama-3.2-1B-Instruct": {"layer": 10, "alpha": 10.0, "hidden": 2048, "vram_gb": 2.0},
    "meta-llama/Prompt-Guard-86M":       {"layer":  0, "alpha":  0.0, "hidden":  768, "vram_gb": 0.5},
}
cfg = _DEFAULTS[SELECTED_MODEL]

os.environ["LOCAL_MODEL_NAME"] = SELECTED_MODEL
os.environ["STEER_LAYER"]      = str(cfg["layer"])
os.environ["STEER_ALPHA"]      = str(cfg["alpha"])

print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
print()
print(f"Selected model : {SELECTED_MODEL}")
print(f"STEER_LAYER    : {cfg['layer']}")
print(f"STEER_ALPHA    : {cfg['alpha']}")
print(f"Hidden dim     : {cfg['hidden']}")
print(f"VRAM needed    : ~{cfg['vram_gb']} GB")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU   : {gpu_name}")
    print(f"VRAM  : {vram_gb:.1f} GB")
    if vram_gb < cfg["vram_gb"]:
        print(f"⚠  Only {vram_gb:.1f} GB VRAM — need ~{cfg['vram_gb']} GB.")
        if "1B" in SELECTED_MODEL:
            print("  1B model can also run on CPU — remove device_map in Cell 6.")
    else:
        print(f"✓ Sufficient VRAM for {SELECTED_MODEL}")
else:
    if "Prompt-Guard" in SELECTED_MODEL:
        print("✓ No GPU needed for Prompt-Guard-86M — CPU is fine.")
    elif "1B" in SELECTED_MODEL:
        print("⚠  No GPU — 1B model will run on CPU (slow but functional).")
    else:
        print("⚠  No GPU — 3B model needs at least a T4. Change runtime type.")


torch        : 2.10.0+cu128
transformers : 5.9.0
CUDA         : True

Selected model : meta-llama/Llama-3.2-3B-Instruct
STEER_LAYER    : 16
STEER_ALPHA    : 12.0
Hidden dim     : 3072
VRAM needed    : ~6.0 GB

GPU   : NVIDIA RTX 6000 Ada Generation
VRAM  : 50.9 GB
✓ Sufficient VRAM for meta-llama/Llama-3.2-3B-Instruct


In [6]:
# ─── Cell 4 — Verify main.py is the Llama-3.2 version ───────────────────────

from pathlib import Path

main_file = Path('main.py')
if not main_file.exists():
    raise FileNotFoundError(
        f'main.py not found in {Path.cwd()}.\n'
        'Upload main.py (the Llama-3.2 version) to the same directory as this notebook.'
    )

content = main_file.read_text()

checks = {
    'Llama-3.2-3B-Instruct in LOCAL_MODEL_NAME default': 'Llama-3.2-3B-Instruct' in content,
    '_resolve_steer_config() present':                   '_resolve_steer_config' in content,
    'Llama-3.2 prompt format (begin_of_text)':           'begin_of_text' in content,
    'Dual EOS token logic (eot_id)':                     'eot_id' in content,
    'add_special_tokens=False':                          'add_special_tokens=False' in content,
    'Prompt-Guard support (run_prompt_guard_check)':     'run_prompt_guard_check' in content,
    '_model_cache dict present':                         '_model_cache' in content,
}

all_ok = True
for desc, result in checks.items():
    icon = '✓' if result else '✗'
    print(f'  {icon} {desc}')
    if not result:
        all_ok = False

if not all_ok:
    raise ValueError(
        'main.py failed one or more checks.\n'
        'Make sure you uploaded the Llama-3.2 version of main.py.'
    )

print(f'\n✓ main.py at {main_file.resolve()} — all checks passed')
print(f'  Size: {main_file.stat().st_size / 1024:.1f} KB')


  ✓ Llama-3.2-3B-Instruct in LOCAL_MODEL_NAME default
  ✓ _resolve_steer_config() present
  ✓ Llama-3.2 prompt format (begin_of_text)
  ✓ Dual EOS token logic (eot_id)
  ✓ add_special_tokens=False
  ✓ Prompt-Guard support (run_prompt_guard_check)
  ✓ _model_cache dict present

✓ main.py at /home/pakdd/github repo/EMNLP 2026/A2A_KG_Llama-3.2-3B-Instruct/main.py — all checks passed
  Size: 100.3 KB


In [7]:
# ─── Cell 5 — Configure environment and import main.py ──────────────────────
#
# STEER_LAYER / STEER_ALPHA were already set in Cell 3.
# This cell imports main.py and verifies the spaCy NER pipeline.

import os, importlib.util
from pathlib import Path

os.environ['STYLE_CACHE_DIR'] = '.style_cache'
os.environ['HF_TOKEN']        = HF_TOKEN
Path('.style_cache').mkdir(exist_ok=True)

spec        = importlib.util.spec_from_file_location('main_module', 'main.py')
main_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(main_module)

print('✓ main.py imported successfully')
print('=' * 60)
print(f'  Model       : {os.environ["LOCAL_MODEL_NAME"]}')
print(f'  Steer layer : {os.environ["STEER_LAYER"]}')
print(f'  Steer alpha : {os.environ["STEER_ALPHA"]}')
print(f'  Cache dir   : {os.environ["STYLE_CACHE_DIR"]}')
print(f'  Contrast pairs: empathetic={len(main_module.CONTRAST_PAIRS["empathetic"])},'
      f' formal={len(main_module.CONTRAST_PAIRS["formal"])}')
print('=' * 60)

nlp_check = main_module._get_nlp()
if nlp_check:
    print('✓ spaCy en_core_web_sm loaded for KG-NER')
else:
    print('⚠  spaCy unavailable — regex-only NER fallback active')

if 'Prompt-Guard' in os.environ['LOCAL_MODEL_NAME']:
    print()
    print('info: Prompt-Guard-86M selected — skip Cells 6-11, go to Cell 12.')


✓ main.py imported successfully
  Model       : meta-llama/Llama-3.2-3B-Instruct
  Steer layer : 16
  Steer alpha : 12.0
  Cache dir   : .style_cache
  Contrast pairs: empathetic=16, formal=16
✓ spaCy en_core_web_sm loaded for KG-NER


In [8]:
# ─── Cell 6 — Load Llama-3.2 model (GPU-first, no CPU offload) ──────────────
#
# Skip this cell if using Prompt-Guard-86M (it loads in Cell 12).
#
# VRAM requirements:
#   Llama-3.2-3B-Instruct : ~6 GB  => T4 (15 GB) or better
#   Llama-3.2-1B-Instruct : ~2 GB  => free T4 or even CPU

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME     = os.environ['LOCAL_MODEL_NAME']
HF_TOKEN_      = os.environ['HF_TOKEN']
cfg            = _DEFAULTS[MODEL_NAME]
MODEL_NEEDS_GB = cfg['vram_gb']

if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_vram_gb  = (torch.cuda.get_device_properties(0).total_memory
                     - torch.cuda.memory_reserved()) / 1e9
    print(f'  GPU total VRAM  : {total_vram_gb:.1f} GB')
    print(f'  VRAM free now   : {free_vram_gb:.1f} GB')
else:
    total_vram_gb = free_vram_gb = 0

if free_vram_gb >= MODEL_NEEDS_GB:
    device_map_arg = {'': 0}
    max_memory_arg = None
    print('  Strategy : GPU-only (device_map={\'\': 0}) — no CPU offload')
elif '1B' in MODEL_NAME:
    device_map_arg = 'cpu'
    max_memory_arg = None
    print('  Strategy : CPU fallback for 1B model')
else:
    gpu_limit      = f'{int(total_vram_gb - 1)}GiB'
    device_map_arg = 'auto'
    max_memory_arg = {0: gpu_limit, 'cpu': '4GiB'}
    print(f'  Strategy : auto split — GPU {gpu_limit} + cpu 4GiB')

print(f'\nLoading {MODEL_NAME} ...')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, token=HF_TOKEN_,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_load_kwargs = dict(
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map=device_map_arg,
    token=HF_TOKEN_,
)
if max_memory_arg is not None:
    _load_kwargs['max_memory'] = max_memory_arg

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **_load_kwargs)
model.eval()

# Inject into main_module cache so build_style_vector() reuses this instance
main_module._model_cache['model']     = model
main_module._model_cache['tokenizer'] = tokenizer

if torch.cuda.is_available() and device_map_arg != 'cpu':
    vram_used = torch.cuda.memory_allocated() / 1e9
    print(f'\n✓ {MODEL_NAME} loaded')
    print(f'  VRAM allocated : {vram_used:.2f} GB')
    cpu_params = sum(1 for p in model.parameters() if p.device.type in ('cpu', 'meta'))
    gpu_params = sum(1 for p in model.parameters() if p.device.type == 'cuda')
    print(f'  GPU params     : {gpu_params}')
    if cpu_params:
        print(f'  ⚠  CPU/meta params: {cpu_params} — restart if errors occur')
    else:
        print('  ✓ All parameters on GPU')
else:
    print(f'\n✓ {MODEL_NAME} loaded on CPU')

n_layers = (len(model.model.layers)
            if hasattr(model, 'model') and hasattr(model.model, 'layers') else '?')
hidden   = model.config.hidden_size if hasattr(model, 'config') else '?'
print(f'  Layers     : {n_layers}  (STEER_LAYER={os.environ["STEER_LAYER"]})')
print(f'  Hidden dim : {hidden}  (expected {cfg["hidden"]} for this model)')

_eot_id = tokenizer.convert_tokens_to_ids('<|eot_id|>')
print(f'  EOS IDs    : [{tokenizer.eos_token_id}, {_eot_id}]  (128001 + 128009)')
print(f'  _model_cache : populated ✓')


  GPU total VRAM  : 50.9 GB
  VRAM free now   : 50.9 GB
  Strategy : GPU-only (device_map={'': 0}) — no CPU offload

Loading meta-llama/Llama-3.2-3B-Instruct ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]


✓ meta-llama/Llama-3.2-3B-Instruct loaded
  VRAM allocated : 6.43 GB
  GPU params     : 254
  ✓ All parameters on GPU
  Layers     : 28  (STEER_LAYER=16)
  Hidden dim : 3072  (expected 3072 for this model)
  EOS IDs    : [128009, 128009]  (128001 + 128009)
  _model_cache : populated ✓


In [9]:
# ─── Cell 7 — Build EMPATHETIC style vector ─────────────────────────────────
# Model already loaded in Cell 6 — no re-download.
# 3B: ~2-4 min on T4   |   1B: ~1-2 min

print('Building EMPATHETIC vector (16 contrast pairs x 2 activations)...')
print()

vec_emp = main_module.build_style_vector(style='empathetic', method='pca')

print(f'\n✓ EMPATHETIC vector built')
print(f'  shape : {list(vec_emp.shape)}')
print(f'  norm  : {vec_emp.norm().item():.6f}  (should be ~1.0)')
print(f'  dtype : {vec_emp.dtype}')


Building EMPATHETIC vector (16 contrast pairs x 2 activations)...


✓ EMPATHETIC vector built
  shape : [3072]
  norm  : 1.000001  (should be ~1.0)
  dtype : torch.float32


In [10]:
# ─── Cell 8 — Build FORMAL style vector ─────────────────────────────────────
# Model already cached from Cell 7 — this cell is fast (~30-60 sec).

print('Building FORMAL vector (model already loaded)...')
print()

vec_form = main_module.build_style_vector(style='formal', method='pca')

print(f'\n✓ FORMAL vector built')
print(f'  shape : {list(vec_form.shape)}')
print(f'  norm  : {vec_form.norm().item():.6f}  (should be ~1.0)')
print(f'  dtype : {vec_form.dtype}')


Building FORMAL vector (model already loaded)...


✓ FORMAL vector built
  shape : [3072]
  norm  : 1.000000  (should be ~1.0)
  dtype : torch.float32


In [11]:
# ─── Cell 9 — Verify both vectors ───────────────────────────────────────────

import torch

_HIDDEN_DIMS = {
    'meta-llama/Llama-3.2-3B-Instruct': 3072,
    'meta-llama/Llama-3.2-1B-Instruct': 2048,
}
HIDDEN_DIM = _HIDDEN_DIMS.get(os.environ['LOCAL_MODEL_NAME'], 3072)

sim = torch.nn.functional.cosine_similarity(
    vec_emp.unsqueeze(0), vec_form.unsqueeze(0)
).item()

print('=' * 55)
print(f'  empathetic : norm={vec_emp.norm():.6f}  shape={list(vec_emp.shape)}')
print(f'  formal     : norm={vec_form.norm():.6f}  shape={list(vec_form.shape)}')
print(f'  cosine sim : {sim:.4f}')
print('=' * 55)

assert abs(vec_emp.norm().item()  - 1.0) < 0.01, f'empathetic not unit-norm! {vec_emp.norm():.4f}'
assert abs(vec_form.norm().item() - 1.0) < 0.01, f'formal not unit-norm! {vec_form.norm():.4f}'
assert vec_emp.shape[0]  == HIDDEN_DIM, f'Expected {HIDDEN_DIM}, got {vec_emp.shape[0]}'
assert vec_form.shape[0] == HIDDEN_DIM, f'Expected {HIDDEN_DIM}, got {vec_form.shape[0]}'
assert sim < 0.5, f'Cosine sim {sim:.3f} too high — vectors may not be distinct!'

print()
if sim < 0:
    print('✓ Cosine similarity NEGATIVE — vectors point in opposite directions (ideal)')
else:
    print(f'  sim={sim:.3f} > 0 — acceptable as long as sim < 0.5.')

import os
cache_files = sorted(os.listdir('.style_cache'))
print('\nCached files in .style_cache/:')
for f in cache_files:
    size = os.path.getsize(f'.style_cache/{f}') / 1024
    print(f'  {f}  ({size:.1f} KB)')
_expected_kb = HIDDEN_DIM * 4 / 1024
print(f'  Note: ~{_expected_kb:.0f} KB each = {HIDDEN_DIM} float32 values')
print('\n✓ All assertions passed!')


  empathetic : norm=1.000001  shape=[3072]
  formal     : norm=1.000000  shape=[3072]
  cosine sim : -0.9709

✓ Cosine similarity NEGATIVE — vectors point in opposite directions (ideal)

Cached files in .style_cache/:
  style_vec_empathetic_pca.pkl  (12.4 KB)
  style_vec_formal_pca.pkl  (12.4 KB)
  Note: ~12 KB each = 3072 float32 values

✓ All assertions passed!


In [12]:
# ─── Cell 10 — Smoke test: activation steering changes tone ─────────────────
#
# Llama-3.2 uses the same prompt format as 3.1:
#   <|begin_of_text|> / <|start_header_id|> / <|eot_id|>
#   add_special_tokens=False
#   eos_token_id = [128001, 128009]

import torch

_, ALPHA_TEST = main_module._resolve_steer_config()
model, tokenizer = main_module._get_model_and_tokenizer()

_eos_ids = [tokenizer.eos_token_id]
_eot_id  = tokenizer.convert_tokens_to_ids('<|eot_id|>')
if _eot_id and _eot_id not in _eos_ids:
    _eos_ids.append(_eot_id)
print(f'EOS token IDs : {_eos_ids}  (128001=<|end_of_text|>, 128009=<|eot_id|>)')

system_msg = (
    'You are a professional customer support agent. '
    'Write a support reply. Do NOT use headers or bullet points. '
    'Write ONLY the reply — 3 to 4 sentences maximum.'
)
test_prompt = (
    '<|begin_of_text|>'
    '<|start_header_id|>system<|end_header_id|>\n\n'
    f'{system_msg}<|eot_id|>'
    '<|start_header_id|>user<|end_header_id|>\n\n'
    'Customer message: My <PRODUCT> order <ORDER_ID> has an <ISSUE>.<|eot_id|>'
    '<|start_header_id|>assistant<|end_header_id|>\n\n'
)

def generate_steered(prompt, style_vec, alpha, label):
    device = next(model.parameters()).device
    vec    = style_vec.to(device=device, dtype=torch.float32)
    layer  = main_module._get_layer(model)

    def _hook(module, inp, output):
        h = output[0] if isinstance(output, tuple) else output
        h_f32   = h.float() + alpha * vec
        steered = h_f32.to(h.dtype)
        return (steered,) + output[1:] if isinstance(output, tuple) else steered

    handle = layer.register_forward_hook(_hook)
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(device)
    try:
        with torch.no_grad():
            out = model.generate(
                inputs.input_ids,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
                eos_token_id=_eos_ids,
            )
    finally:
        handle.remove()

    new_ids = out[0][inputs.input_ids.shape[1]:]
    text    = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    print(f'\n[{label}]\n  {text[:300]}')
    return text

print('=' * 60)
print('Smoke Test: same prompt, two steering vectors, different tones')
print('=' * 60)

r_emp  = generate_steered(test_prompt, vec_emp,  ALPHA_TEST, 'EMPATHETIC')
r_form = generate_steered(test_prompt, vec_form, ALPHA_TEST, 'FORMAL')

print()
print('=' * 60)
print('✓ Smoke test complete')
print('  EMPATHETIC -> warm personal language ("I\'m so sorry", "I understand")')
print('  FORMAL     -> institutional language ("We acknowledge", "has been logged")')
print('=' * 60)


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


EOS token IDs : [128009]  (128001=<|end_of_text|>, 128009=<|eot_id|>)
Smoke Test: same prompt, two steering vectors, different tones


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



[EMPATHETIC]
  Dear Customer, 
Thank for contacting our company regarding your product order with Order ID number . We apologize for any inconvenience caused due to issue in delivery of products as stated by customers . Further information about this issue will be required from the user at present time .
Please re

[FORMAL]
  I'm so sorry I can see that you're having some issues with your Order #12345, and let me get started on getting us sorted out for you right away! Can you tell me exactly what's happening - is it something about my product itself? And just one more thing, did we have someone come over here today beca

✓ Smoke test complete
  EMPATHETIC -> warm personal language ("I'm so sorry", "I understand")
  FORMAL     -> institutional language ("We acknowledge", "has been logged")


In [13]:
# ─── Cell 11 — Verify saved .pkl files and next steps ───────────────────────

import os, pickle, torch
from pathlib import Path

cache_dir = Path('.style_cache')
pkl_files = sorted(cache_dir.glob('*.pkl'))

print('=' * 60)
print('Stored vectors in .style_cache/')
print('=' * 60)

if not pkl_files:
    print('  (no .pkl files found — did Cells 7 and 8 run successfully?)')
else:
    for pkl_path in pkl_files:
        size_kb = pkl_path.stat().st_size / 1024
        with open(pkl_path, 'rb') as f:
            v = pickle.load(f)
        print(f'  {pkl_path.name:<45} {size_kb:6.2f} KB  '
              f'shape={list(v.shape)}  norm={v.norm():.4f}')

print('=' * 60)
print(f'Cache dir (absolute): {cache_dir.resolve()}')
print()
_model  = os.environ['LOCAL_MODEL_NAME']
_hidden = {'meta-llama/Llama-3.2-3B-Instruct': 3072,
           'meta-llama/Llama-3.2-1B-Instruct': 2048}.get(_model, 3072)
print(f'  Each file is ~{_hidden*4//1024} KB — {_hidden} float32 values ({_model})')
print()
print('Next steps:')
print('  1. Download the .style_cache/ folder')
print('  2. Place it next to main.py')
print('  3. Set in .env:')
print(f'       LOCAL_MODEL_NAME={_model}')
print(f'       STEER_LAYER={os.environ["STEER_LAYER"]}')
print(f'       STEER_ALPHA={os.environ["STEER_ALPHA"]}')
print('       STYLE_CACHE_DIR=.style_cache')
print('       HF_TOKEN=hf_...')
print('  4. Run: python main.py battery_issue')
print()
print('✓ Vector build pipeline complete!')


Stored vectors in .style_cache/
  style_vec_empathetic_pca.pkl                   12.39 KB  shape=[3072]  norm=1.0000
  style_vec_formal_pca.pkl                       12.39 KB  shape=[3072]  norm=1.0000
Cache dir (absolute): /home/pakdd/github repo/EMNLP 2026/A2A_KG_Llama-3.2-3B-Instruct/.style_cache

  Each file is ~12 KB — 3072 float32 values (meta-llama/Llama-3.2-3B-Instruct)

Next steps:
  1. Download the .style_cache/ folder
  2. Place it next to main.py
  3. Set in .env:
       LOCAL_MODEL_NAME=meta-llama/Llama-3.2-3B-Instruct
       STEER_LAYER=16
       STEER_ALPHA=12.0
       STYLE_CACHE_DIR=.style_cache
       HF_TOKEN=hf_...
  4. Run: python main.py battery_issue

✓ Vector build pipeline complete!


In [15]:
# ─── Cell 13 — [Optional] Run the full A2A pipeline (single scenario) ────────
# Requires GROQ_API_KEY. Not applicable for Prompt-Guard-86M.
# Scenarios: battery_issue | wrong_item | billing_error | delivery_delay

import asyncio

SCENARIO = "battery_issue"   # change to try other scenarios

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "gsk_YOUR_KEY_HERE":
    print("⚠  Skipping: GROQ_API_KEY not set. Fill it in Cell 2.")
elif 'Prompt-Guard' in os.environ.get('LOCAL_MODEL_NAME', ''):
    print("⚠  Prompt-Guard-86M cannot generate text — switch to 3B or 1B for this cell.")
else:
    from langchain_groq import ChatGroq
    llm          = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
    llm_customer = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        import subprocess; subprocess.run(["pip", "install", "-q", "nest_asyncio"], check=True)
        import nest_asyncio; nest_asyncio.apply()
    print(f"Running pipeline — scenario: {SCENARIO}")
    record = asyncio.run(main_module.run_pipeline(SCENARIO, llm, llm_customer))
    emp_out = {"kg_summary": record["knowledge_graph"],
               "defactualized_prompt": record["empathetic_output"]["defactualized_prompt"]}
    main_module.console.print(main_module.make_kg_panel(emp_out))
    emp_response  = {**record["empathetic_output"], "kg_summary": record["knowledge_graph"]}
    form_response = {**record["formal_output"],     "kg_summary": record["knowledge_graph"]}
    main_module.console.print(main_module.make_comparison_table(emp_response, form_response))
    print(f"\nFull record keys: {list(record.keys())}")


Running pipeline — scenario: battery_issue


╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Priya Sharma   Order: ORD-7741                                                                        │
│ Product: TechPro X200 Laptop                                                                                    │
│ Issue: battery draining completely within 2 hours of a full charge                                              │
│ Sentiment: FRUSTRATED   Urgency: high                                                                           │
│                                                                                                                 │
│ "I'm reaching out again regarding my TechPro X200 Laptop, order number ORD-7741, which I purchased just three   │
│ weeks ago. As I mentioned in my previous support request, the battery is draining extremely quickly - it's      │
│ completely dead within 2 hours of a full charge, which is making it impossible for me to meet my deadlines.     │
│ Despite updating the drivers as suggested last week, the issue persists, and I have a critical client project   │
│ due tomorrow, so I urgently need a more effective solution today. Could you please assist me in resolving this  │
│ issue as soon as possible?"                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────── Empathetic — Support Response ───────────────────╮
│ Style: EMPATHETIC   Steering mode: kg_activation                   │
│                                                                    │
│ Final response (rehydrated):                                       │
│ It sounds like we've already taken some steps with driver updates  │
│ but unfortunately that hasn't resolved your issues yet, so let us  │
│ try another approach together to get your product working smoothly │
│ once more. Can you tell me what kind of work do you need the       │
│ device to perform on an ongoing basis? This will help our          │
│ technical team provide further assistance accordingly.             │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ It sounds like we've already taken some steps with driver updates  │
│ but unfortunately that hasn't resolved your issues yet, so let us  │
│ try another approach together to get your product working smoothly │
│ o                                                                  │
╰────────────────────────────────────────────────────────────────────╯
╭──────────────────── Formal — Support Response ─────────────────────╮
│ Style: FORMAL   Steering mode: kg_activation                       │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Our technical team will investigate the matter further and provide │
│ an update on possible solutions by return email today. In the      │
│ meantime, we recommend keeping all relevant documentation related  │
│ to our conversation together with your product packaging for any   │
│ future reference. We appreciate your patience while we work        │
│ towards finding a resolution to this inconvenience caused.         │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Our technical team will investigate the matter further and provide │
│ an update on possible solutions by return email today. In the      │
│ meantime, we recommend keeping all relevant documentation related  │
│ to ou                                                              │
╰────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Knowledge Graph ────────────────────────────────────────────────╮
│ Nodes (7):                                                                                                      │
│   (PRODUCT) TechPro X200 Laptop  (salience: 0.8)                                                                │
│   (ORDER_ID) ORD-7741  (salience: 0.9)                                                                          │
│   (ISSUE) persists, and I have a critical client project due tomorrow, so I urgently need a more effective      │
│ solution today  (salience: 0.9)                                                                                 │
│   (ISSUE) as soon as possible  (salience: 0.9)                                                                  │
│   (URGENCY) high  (salience: 1.0)                                                                               │
│   (SENTIMENT) polite  (salience: 0.7)                                                                           │
│   (CUSTOMER_NAME) Priya Sharma  (salience: 0.7)                                                                 │
│                                                                                                                 │
│ Edges (6):                                                                                                      │
│   TechPro X200 Laptop       —has_issue→ persists, and I have a cr                                               │
│   TechPro X200 Laptop       —has_issue→ as soon as possible                                                     │
│   ORD-7741                  —about_product→ TechPro X200 Laptop                                                 │
│   Priya Sharma              —placed_order→ ORD-7741                                                             │
│   persists, and I have a cr —has_urgency→ high                                                                  │
│   as soon as possible       —has_urgency→ high                                                                  │
│                                                                                                                 │
│ Defactualized prompt:                                                                                           │
│   I'm reaching out again regarding my <PRODUCT>, order number <ORDER_ID>, which I purchased just three weeks    │
│ ago. As I mentioned in my previous support request, the battery is draining extremely quickly                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                   Style Comparison — Same Facts, Different Tone                                   
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Dimension       ┃ Empathetic Pipeline                           ┃ Formal Pipeline                               ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Target style    │ empathetic                                    │ formal                                        │
│ Steering mode   │ kg_activation                                 │ kg_activation                                 │
│ KG nodes        │ 7                                             │ 7                                             │
│ KG edges        │ 6                                             │ 6                                             │
│ Response (first │ It sounds like we've already taken some steps │ Our technical team will investigate the       │
│ 200 chars)      │ with driver updates but unfortunately that    │ matter further and provide an update on       │
│                 │ hasn't resolved your issues yet, so let us    │ possible solutions by return email today. In  │
│                 │ try another approach together to get your     │ the meantime, we recommend keeping all        │
│                 │ product working smoothly o                    │ relevant documentation related to ou          │
└─────────────────┴───────────────────────────────────────────────┴───────────────────────────────────────────────┘


Full record keys: ['input', 'knowledge_graph', 'empathetic_output', 'formal_output']


In [16]:
# ─── Cell 14 — [Optional] Run the full 100-case batch ───────────────────────
# ⚠  ~15–40 min depending on Groq rate limits.

import asyncio
from pathlib import Path

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "gsk_YOUR_KEY_HERE":
    print("⚠  Skipping: GROQ_API_KEY not set.")
elif 'Prompt-Guard' in os.environ.get('LOCAL_MODEL_NAME', ''):
    print("⚠  Prompt-Guard-86M cannot run the batch pipeline — switch to 3B or 1B.")
else:
    _model_slug = os.environ['LOCAL_MODEL_NAME'].split('/')[-1].lower().replace('-', '_')
    out_dir = Path(f'outputs_{_model_slug}')
    out_dir.mkdir(exist_ok=True)
    (out_dir / 'knowledge_graphs').mkdir(exist_ok=True)
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        import subprocess; subprocess.run(['pip', 'install', '-q', 'nest_asyncio'], check=True)
        import nest_asyncio; nest_asyncio.apply()
    print(f'Starting batch — {len(main_module.BATCH_CASES)} cases — output: {out_dir}')
    asyncio.run(main_module.run_batch())


Starting batch — 100 cases — output: outputs_llama_3.2_3b_instruct


──────────────────────────────────── A2A Customer Support — 100-Case Batch Run ────────────────────────────────────

Output file : outputs/results_20260524_062711.jsonl
  Total cases : 100
  Activation steering: ENABLED

──────────────── Case   1/100  scenario=battery_issue  customer=Priya Sharma  sentiment=frustrated ────────────────

╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Priya Sharma   Order: ORD-7741                                                                        │
│ Product: TechPro X200 Laptop                                                                                    │
│ Issue: battery draining completely within 2 hours of a full charge                                              │
│ Sentiment: FRUSTRATED   Urgency: high                                                                           │
│                                                                                                                 │
│ "I'm reaching out again regarding my TechPro X200 Laptop, order number ORD-7741, which I purchased just three   │
│ weeks ago. As I mentioned in my previous support request, the battery is draining extremely quickly - it's now  │
│ completely dead within 2 hours of a full charge, which is really impacting my work. I followed the advice from  │
│ my last support interaction and updated the drivers, but unfortunately, the issue persists. I have a critical   │
│ deadline to meet tomorrow and I urgently need a solution to this problem, could you please assist me in         │
│ resolving this today?"                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Ximena Flores   Order: ORD-8858                                                                       │
│ Product: Ergonomic Office Chair                                                                                 │
│ Issue: delivery now 3 weeks overdue with no update from courier                                                 │
│ Sentiment: NEUTRAL   Urgency: medium                                                                            │
│                                                                                                                 │
│ "I'm reaching out about my order, ORD-8858, for the Ergonomic Office Chair, which is now three weeks past the   │
│ expected delivery date. Unfortunately, I haven't received any updates from the courier, and I'm still waiting   │
│ to receive my chair. I've been working on the floor for weeks, so I'd appreciate it if you could provide a      │
│ clear timeline for when I can expect to receive my order or offer a refund so I can explore alternative options │
│ locally. Could you please look into this and let me know the next steps?"                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────── Empathetic — Support Response ───────────────────╮
│ Style: EMPATHETIC   Steering mode: kg_activation                   │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Please rest assured that we're taking immediate action to locate   │
│ your missing package and ensure its safe arrival as soon as        │
│ possible - our team will also contact the courier directly to get  │
│ an update on their end. In the meantime, I'll go ahead and process │
│ a full refund of $X.XX towards your original payment method,       │
│ allowing you to consider other local alternatives while we work on │
│ resolving this issue further with the courier. You should see the  │
│ credit applied within [timeframe] days.                            │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Please rest assured that we're taking immediate action to locate   │
│ your missing package and ensure its safe arrival as soon as        │
│ possible - our team will also contact the courier directly to get  │
│ an update                                                          │
╰────────────────────────────────────────────────────────────────────╯
╭──────────────────── Formal — Support Response ─────────────────────╮
│ Style: FORMAL   Steering mode: kg_activation                       │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Our team will investigate with our shipping partner immediately    │
│ and contact you within the next two business days regarding an     │
│ estimated update on the status of your shipment. In the meantime,  │
│ we would like to inform you that due to unforeseen circumstances   │
│ beyond our control, delays may have occurred in processing and     │
│ dispatching orders; however, we assure you that every effort will  │
│ be made to expedite its resolution as soon as possible. We kindly  │
│ request patience while we work towards resolving this matter       │
│ further.                                                           │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Our team will investigate with our shipping partner immediately    │
│ and contact you within the next two business days regarding an     │
│ estimated update on the status of your shipment. In the meantime,  │
│ we woul                                                            │
╰────────────────────────────────────────────────────────────────────╯

✓ Saved case 100  |  KG → knowledge_graphs/kg_100_delivery_delay_ORD-8858_Ximena_Flores.json

───────────────────────────────────────────────── Batch complete ──────────────────────────────────────────────────

Success: 100/100
  Failed:  0/100
  Results: outputs/results_20260524_062711.jsonl
  KGs:     outputs/knowledge_graphs/  (100 files)